# Qwen2.5-VL Benchmark Fine-Tuning in Colab

This notebook runs the generic benchmark fine-tuning workflow in `fine-tuning/` using a Google Colab GPU runtime. The default configuration is a small Fashion-MNIST smoke test: training data is exported from `train`, and evaluation uses the held-out `test` split.

Before running: select **Runtime > Change runtime type > GPU**. The repository branch configured below must already contain `fine-tuning/prepare_benchmark.py`, `fine-tuning/train_qwen25vl_benchmark.py`, and `fine-tuning/evaluate_qwen25vl_benchmark.py`.

## Configuration

The defaults provide a functional pipeline check rather than a serious fine-tuning experiment. Increase `TRAIN_EXAMPLES`, `VALIDATION_EXAMPLES`, `EPOCHS`, and `EVAL_SAMPLES` only after the smoke run completes.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/samhatescoding/transformers.git"
BRANCH = "main"

BENCHMARK = "fashion_mnist"
TRAIN_SPLIT = "train"
EVAL_SPLIT = "test"
TRAIN_EXAMPLES = 20
VALIDATION_EXAMPLES = 10
EVAL_SAMPLES = 20
EPOCHS = 1
LEARNING_RATE = "1e-4"
GRADIENT_ACCUMULATION_STEPS = 4

RUN_TRAINING = True
RUN_BASE_EVALUATION = True
RUN_ADAPTER_EVALUATION = True

DATA_DIR = Path(f"fine-tuning/data/{BENCHMARK}_generic_smoke")
ADAPTER_DIR = Path(f"fine-tuning/output/qwen2.5-vl-3b-{BENCHMARK}-generic-smoke-lora")
BASE_MODEL_NAME = f"qwen2.5-vl-3b-{BENCHMARK}-base-smoke"
ADAPTER_MODEL_NAME = f"qwen2.5-vl-3b-{BENCHMARK}-generic-smoke-lora"

print("Benchmark:", BENCHMARK)
print("Training split:", TRAIN_SPLIT, "Evaluation split:", EVAL_SPLIT)
print("Adapter output:", ADAPTER_DIR)

## Clone Repository And Install Dependencies

In [ ]:
import os
import shutil
import subprocess

repo_dir = Path("/content/transformers")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
print("Working directory:", Path.cwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True, cwd=repo_dir)

def run_repo_command(command):
    """Run a repository CLI command from the clone, even if Colab changes cwd."""
    print("$", " ".join(str(part) for part in command), flush=True)
    result = subprocess.run(command, cwd=repo_dir)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit status {result.returncode}. "
            "Read the command output immediately above this error. "
            "If a file is missing, rerun the clone/setup cells first."
        )
    return result

In [ ]:
import torch

!nvidia-smi
%pip install -q -r requirements.txt -r fine-tuning/requirements-qwen.txt
%pip uninstall -y torchao

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU runtime before continuing.")

## Verify The Generic Scripts

This checks that the cloned commit includes the generic fine-tuning files and that their focused unit tests pass.

In [ ]:
required_scripts = [
    Path("fine-tuning/prepare_benchmark.py"),
    Path("fine-tuning/train_qwen25vl_benchmark.py"),
    Path("fine-tuning/evaluate_qwen25vl_benchmark.py"),
]
missing = [str(path) for path in required_scripts if not path.exists()]
if missing:
    raise FileNotFoundError(f"Push the generic fine-tuning files before using this notebook. Missing: {missing}")

!python fine-tuning/prepare_benchmark.py --help
!python -m unittest tests.test_fine_tuning tests.test_benchmark_runner

## Export Training And Validation Data

The default invocation takes separate non-overlapping slices from Fashion-MNIST `train`. It does not export examples from the held-out `test` evaluation split.

In [ ]:
prepare_command = [
    "python", "fine-tuning/prepare_benchmark.py",
    "--benchmark", BENCHMARK,
    "--train-split", TRAIN_SPLIT,
    "--train-examples", str(TRAIN_EXAMPLES),
    "--validation-examples", str(VALIDATION_EXAMPLES),
    "--output-dir", str(DATA_DIR),
]
run_repo_command(prepare_command)

In [ ]:
import json
from IPython.display import display
from PIL import Image

resolved_data_dir = repo_dir / DATA_DIR
metadata = json.loads((resolved_data_dir / "metadata.json").read_text(encoding="utf-8"))
train_record = json.loads((resolved_data_dir / "train_manifest.jsonl").read_text(encoding="utf-8").splitlines()[0])
validation_record = json.loads((resolved_data_dir / "validation_manifest.jsonl").read_text(encoding="utf-8").splitlines()[0])
print(json.dumps(metadata, indent=2))
print("First training target:", train_record["answer"])
print("First validation target:", validation_record["answer"])
display(Image.open(resolved_data_dir / train_record["image_path"]))

## Fine-Tune A Small Adapter

This cell downloads Qwen2.5-VL-3B-Instruct and performs QLoRA training. The default dataset is intentionally small, so completion validates the pipeline but the adapter is not expected to outperform the base model consistently.

In [ ]:
training_command = [
    "python", "fine-tuning/train_qwen25vl_benchmark.py",
    "--train-manifest", str(DATA_DIR / "train_manifest.jsonl"),
    "--validation-manifest", str(DATA_DIR / "validation_manifest.jsonl"),
    "--output-dir", str(ADAPTER_DIR),
    "--epochs", str(EPOCHS),
    "--learning-rate", LEARNING_RATE,
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
]
if RUN_TRAINING:
    run_repo_command(training_command)
else:
    print("Training skipped. Set RUN_TRAINING = True in the configuration cell to run it.")

## Evaluate On Held-Out Examples

The base model and adapted model are evaluated on the same examples from `EVAL_SPLIT`. For the default configuration, this is Fashion-MNIST `test`, which was not used to create the adapter.

In [ ]:
base_eval_command = [
    "python", "fine-tuning/evaluate_qwen25vl_benchmark.py",
    "--benchmark", BENCHMARK,
    "--split", EVAL_SPLIT,
    "--model-name", BASE_MODEL_NAME,
    "--samples", str(EVAL_SAMPLES),
]
if RUN_BASE_EVALUATION:
    run_repo_command(base_eval_command)
else:
    print("Base evaluation skipped.")

In [ ]:
adapter_eval_command = [
    "python", "fine-tuning/evaluate_qwen25vl_benchmark.py",
    "--benchmark", BENCHMARK,
    "--split", EVAL_SPLIT,
    "--adapter-path", str(ADAPTER_DIR),
    "--model-name", ADAPTER_MODEL_NAME,
    "--samples", str(EVAL_SAMPLES),
]
if RUN_ADAPTER_EVALUATION:
    if not (repo_dir / ADAPTER_DIR).exists():
        raise FileNotFoundError(f"Adapter not found: {ADAPTER_DIR}. Run training first.")
    run_repo_command(adapter_eval_command)
else:
    print("Adapter evaluation skipped.")

## Summarize Results And Download Artifacts

In [ ]:
from collections import Counter

result_dir = repo_dir / "results/fine-tuning"
result_paths = [
    result_dir / f"{BASE_MODEL_NAME}_{BENCHMARK}.json",
    result_dir / f"{ADAPTER_MODEL_NAME}_{BENCHMARK}.json",
]
for path in result_paths:
    if not path.exists():
        print("Missing result:", path)
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows = payload["report"]["results"]
    correct = sum(bool(row["correct"]) for row in rows)
    predictions = Counter(row["prediction"].strip().lower() for row in rows)
    print(f"{payload['model']}: {correct}/{len(rows)} = {correct / len(rows):.1%}")
    print("Predictions:", dict(predictions))

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive("fine_tuning_smoke_artifacts", "zip", root_dir=repo_dir / "results/fine-tuning")
print("Downloading evaluation results:", archive_path)
files.download(archive_path)

# Uncomment to download the trained adapter as well; this archive is substantially larger.
# adapter_archive = shutil.make_archive("qwen_adapter", "zip", root_dir=repo_dir / ADAPTER_DIR)
# files.download(adapter_archive)

## Trying Another Benchmark

After the default smoke test completes, change the configuration cell. A leakage-free captioning example is:

```python
BENCHMARK = "flickr30k"
TRAIN_SPLIT = "train"
EVAL_SPLIT = "test"
TRAIN_EXAMPLES = 300
VALIDATION_EXAMPLES = 100
EVAL_SAMPLES = 50
EPOCHS = 2
```

Some benchmarks require Hugging Face authentication, gated-dataset approval, or a dataset identifier/split adjustment. Do not train on the same examples used for evaluation.